# 3. Session-scoped runtime limits and the audit log

**What you'll learn:** how AgentVanguard tracks a whole *conversation*
(not just one isolated call), enforces hard limits on it (e.g. "the same
tool call repeated too many times"), and how to read back the durable audit
trail it writes for every decision.

## Prerequisites

Same as notebooks 1-2: `pip install -e ".[notebooks]"`, no network/API
keys needed.

## Why session tracking matters

Every `authorize()` call can carry a `session_id` string. AgentVanguard
uses it to accumulate per-conversation counters -- total steps, total tool
calls, and (the one we'll trigger below) how many times the *exact same*
tool call has been repeated. This catches a confused or looping agent even
if the LLM itself never "decides" anything is wrong -- it's a hard backstop
that doesn't depend on the agent's own judgment.

The limits live under `runtime_limits:` in config
(`examples/agentvanguard.yaml`); we'll temporarily lower
`max_repeated_calls` so this notebook doesn't need dozens of calls to
demonstrate the deny.

## Why the audit log matters

Every single `authorize()` call -- allowed or denied -- is written as one
JSON line to the audit log (`audit.path` in config), tagged with a unique
`trace_id`. This is what lets you reconstruct, after the fact, exactly what
AgentVanguard decided and why for any past request -- the same lookup the
`agentvanguard explain <trace_id>` CLI command performs.

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from agentvanguard import ActionType, AgentVanguard, AgentIdentity, RiskTier, SideEffect, ToolManifest
from agentvanguard.core.errors import VanguardBlockedError

# Use a dedicated audit log for this notebook so re-running it doesn't mix
# records with previous runs. `examples/notebooks/logs/` is already covered
# by the repo's `logs/`/`*.jsonl` .gitignore rules, so nothing here gets
# committed.
AUDIT_PATH = Path.cwd() / "logs" / "session_demo_audit.jsonl"
AUDIT_PATH.parent.mkdir(exist_ok=True)
AUDIT_PATH.unlink(missing_ok=True)

import yaml

from agentvanguard.config.schema import Config

CONFIG_PATH = REPO_ROOT / "examples" / "agentvanguard.yaml"
config_dict = yaml.safe_load(CONFIG_PATH.read_text())
config_dict["audit"]["path"] = str(AUDIT_PATH)
# Lower the repeat limit (default is 3) purely so this demo doesn't need as
# many calls -- everything else is the unmodified example policy.
config_dict["runtime_limits"]["max_repeated_calls"] = 3

firewall = AgentVanguard(Config.model_validate(config_dict))

SUPPORT_AGENT = AgentIdentity(name="support_agent", workflow="customer_support")
SESSION_ID = "notebook-session-demo"
CONTEXT = {"agent": SUPPORT_AGENT, "session_id": SESSION_ID}


## 3.1 Tripping `max_repeated_calls`

We register the same `lookup_order` tool from notebook 2, then call it
*five* times in a row with identical arguments inside the same
`session_id` -- simulating an agent stuck in a loop. With
`max_repeated_calls: 3`, the 4th identical call (and every one after it)
is denied.

In [2]:
@firewall.tool(
    manifest=ToolManifest(
        name="lookup_order",
        action_type=ActionType.READ,
        side_effect=SideEffect.INTERNAL_READ,
        risk_tier=RiskTier.LOW,
        allowed_callers=["support_agent"],
    )
)
def lookup_order(order_id: str) -> str:
    return f"Order {order_id}: status=delivered"


denial_trace_id = None
for attempt in range(1, 6):
    try:
        result = lookup_order(order_id="1234", _context=CONTEXT)
        print(f"attempt {attempt}: allowed -> {result}")
    except VanguardBlockedError as exc:
        print(f"attempt {attempt}: blocked -> {exc.decision.decision.value} ({exc.decision.reasons[0]})")
        denial_trace_id = exc.decision.trace_id

print("\ndenied trace_id:", denial_trace_id)


attempt 1: allowed -> Order 1234: status=delivered
attempt 2: allowed -> Order 1234: status=delivered
attempt 3: allowed -> Order 1234: status=delivered
attempt 4: blocked -> DENY (tool 'lookup_order' called with identical arguments 4 times, exceeding max_repeated_calls 3)
attempt 5: blocked -> DENY (tool 'lookup_order' called with identical arguments 4 times, exceeding max_repeated_calls 3)

denied trace_id: 9852df6a-c9c7-4b4f-aa66-eaed174b6569


## 3.2 Reading the audit log

Each line of the JSONL audit file is one `AuditEvent`: which boundary was
being checked (`action`), which tool (if any), the resulting `decision`,
and the full list of `findings` that led to it. Let's list every event
written for this session, in order.

In [3]:
events = [json.loads(line) for line in AUDIT_PATH.read_text().splitlines() if line.strip()]
print(f"{len(events)} audit events written for session {SESSION_ID!r}\n")

for event in events:
    tool = event["payload"].get("tool")
    print(f"[{event['action']:10s}] tool={tool!r:16} -> {event['decision']:7} (trace_id={event['trace_id']})")


8 audit events written for session 'notebook-session-demo'

[tool_call ] tool='lookup_order'   -> ALLOW   (trace_id=6fb3d490-91ef-4553-9fe7-7808810c7d4d)
[output    ] tool=None             -> ALLOW   (trace_id=901b8baf-fdb8-401b-9ee6-e4a646c27dd1)
[tool_call ] tool='lookup_order'   -> ALLOW   (trace_id=b64fceff-1572-46ce-91c7-2d68b7bfb9e2)
[output    ] tool=None             -> ALLOW   (trace_id=269616b3-9516-4c6a-8d4f-87dfbe2aafc3)
[tool_call ] tool='lookup_order'   -> ALLOW   (trace_id=d329c460-47ee-4222-b29c-ea0ecdd77def)
[output    ] tool=None             -> ALLOW   (trace_id=f4d0f4cf-ffd3-4e26-a24c-dce5a9b85b56)
[tool_call ] tool='lookup_order'   -> DENY    (trace_id=47287337-6d79-4c3f-ba43-53d0332ff219)
[tool_call ] tool='lookup_order'   -> DENY    (trace_id=9852df6a-c9c7-4b4f-aa66-eaed174b6569)


## 3.3 Looking up one specific decision by `trace_id`

This is the part that mirrors `agentvanguard explain <trace_id>` from the
command line: given the `trace_id` of the denial we hit above, find its
exact audit record -- including the structured `findings` (not just the
human-readable `reasons` string) and the original request `payload`.

In [4]:
matching = [e for e in events if e["trace_id"] == denial_trace_id]
print(json.dumps(matching[0], indent=2))


{
  "event_id": "96525f71-a177-4f01-b278-07103a735210",
  "event_type": "authorization",
  "timestamp": "2026-06-20T07:40:52.389632+00:00",
  "trace_id": "9852df6a-c9c7-4b4f-aa66-eaed174b6569",
  "session_id": "notebook-session-demo",
  "action": "tool_call",
  "decision": "DENY",
  "risk_score": 0.8,
  "findings": [
    {
      "type": "repetition_loop",
      "detector": "session_guard:repetition",
      "severity": "high",
      "message": "tool 'lookup_order' called with identical arguments 4 times, exceeding max_repeated_calls 3",
      "span": null
    }
  ],
  "reasons": [
    "tool 'lookup_order' called with identical arguments 4 times, exceeding max_repeated_calls 3"
  ],
  "payload": {
    "action": "tool_call",
    "agent": {
      "name": "support_agent",
      "workflow": "customer_support"
    },
    "tool": "lookup_order",
    "arguments": {
      "order_id": "1234"
    },
    "session_id": "notebook-session-demo"
  }
}


## What's next

**`04_full_agent_loop.ipynb`** ties `wrap_agent` (ingress/egress) and
`wrap_tool` (per-call mediation, including the limits we just saw) together
in one mock multi-turn conversation -- the closure of this whole series.